# 03. Градиентный бустинг: подбор гиперпараметров, повторы, протокол Kaggle

**Цель:** получить *настроенную* модель градиентного бустинга и окончательные строки таблицы
результатов с оценкой разброса, а также честно сравниться с лидербордом HiggsML.

Что делает ноутбук:
1. Подбор гиперпараметров XGBoost (Optuna, TPE) по AMS на val — test не участвует.
2. То же для LightGBM (если установлен) — вторая реализация бустинга для контроля.
3. Переобучение лучшей модели, графики для главы 3, проверка переобучения (KS-тест).
4. Кривая обучения: AMS от объёма обучающей выборки.
5. Пять повторов с разными разбиениями для всех методов → среднее ± σ.
6. Протокол Kaggle: обучение на `KaggleSet == 't'`, оценка на `'v'` с `KaggleWeight`.
7. Важность признаков (gain) — превью для `05_shap`.

**Время работы.** Одно обучение XGBoost на 491 тыс. событий: ~0,5–1,5 мин на CPU ноутбука,
секунды на GPU. Весь ноутбук при N_TRIALS = 30: около часа на CPU, 10–15 минут на GPU.
Для проверки, что всё запускается: `THESIS_FAST=1 jupyter lab` (3 пробы, 2 повтора).

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message=".*IProgress.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import load_higgsml, find_data_file, FEATURES, split_train_val_test, expected_yields
from src.metrics import ams, z0, weighted_roc, significance_scan, ks_overtraining
from src.plots import setup_style, plot_roc, plot_significance_scan, plot_classifier_output
from src.baselines import (cut_based, fit_logreg, fit_xgb, fit_lgb, xgb_device, XGB_DEFAULT,
                           summarize_repeats, xy)

setup_style()
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
DATA_PATH = find_data_file(ROOT / "data")
print("Файл данных:", DATA_PATH.name)
if "synthetic" in DATA_PATH.name:
    print("ВНИМАНИЕ: это синтетические данные для проверки кода, а не физика!")

# ---- Настройки эксперимента -------------------------------------------------
FAST = os.environ.get("THESIS_FAST") == "1"      # быстрая проверка кода: THESIS_FAST=1 jupyter lab
N_TRIALS = 3 if FAST else 30                     # число проб Optuna для XGBoost
N_TRIALS_LGB = 2 if FAST else 20                 # для LightGBM
N_REPEATS = 2 if FAST else 5                     # повторов с разными seed
SEEDS = list(range(N_REPEATS))
DEVICE = xgb_device()                            # 'cuda', если есть GPU и сборка с CUDA
print(f"XGBoost device = {DEVICE};  N_TRIALS = {N_TRIALS};  N_REPEATS = {N_REPEATS};  FAST = {FAST}")

## 1. Данные и разбиение

То же разбиение 60/20/20 с seed = 42, что и в `02_baseline` — результаты сопоставимы.

In [ ]:
df = load_higgsml(DATA_PATH)
train, val, test = split_train_val_test(df, test_size=0.2, val_size=0.2, seed=42)
for name, part in (("train", train), ("val", val), ("test", test)):
    yl = expected_yields(part, "w")
    print(f"{name:5s}: строк {len(part):>8,}   s = {yl['s']:8.1f}   b = {yl['b']:12.1f}")

## 2. Подбор гиперпараметров XGBoost (Optuna)

**Целевая функция** — AMS на val при лучшем пороге (не менее 200 отобранных событий).
Число деревьев не подбирается: ставим 2000 и используем раннюю остановку по logloss на val.
Пространство поиска — стандартное для бустинга:

| параметр | диапазон | смысл |
|---|---|---|
| learning_rate | 0,01–0,3 (лог.) | шаг градиентного спуска |
| max_depth | 3–10 | глубина дерева, нелинейность взаимодействий |
| min_child_weight | 1–100 (лог.) | минимальная сумма весов в листе — регуляризация |
| subsample, colsample_bytree | 0,5–1,0 | доля событий / признаков на дерево |
| reg_lambda | 0,01–10 (лог.) | L2-регуляризация листьев |
| gamma | 0–5 | минимальный выигрыш для разбиения |

Каждая проба пишется в `results/optuna_xgb_trials.csv`, так что прерванный поиск не пропадает.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def xgb_space(trial):
    return dict(
        n_estimators=2000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 100.0, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
    )

def objective_xgb(trial):
    res = fit_xgb(train, val, None, xgb_space(trial), device=DEVICE, seed=42, return_model=False)
    trial.set_user_attr("AUC_val", res["AUC_val"])
    trial.set_user_attr("best_iteration", res["best_iteration"])
    trial.set_user_attr("fit_seconds", res["fit_seconds"])
    return res["AMS_val"]

def report(study, trial):
    best = study.best_value
    print(f"проба {trial.number + 1:3d}/{N_TRIALS}: AMS_val = {trial.value:.3f}  "
          f"(лучшая {best:.3f}), деревьев {trial.user_attrs['best_iteration']}, "
          f"{trial.user_attrs['fit_seconds']:.0f} с")
    study.trials_dataframe().to_csv(RESULTS / "optuna_xgb_trials.csv", index=False)

study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
                                study_name="xgb_ams")
t0 = time.time()
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, callbacks=[report])
print(f"\nГотово за {(time.time() - t0) / 60:.1f} мин. Лучший AMS_val = {study_xgb.best_value:.3f}")
best_xgb = study_xgb.best_params
best_xgb

In [ ]:
# Ход поиска: лучшая AMS по пробам и важность гиперпараметров
tr = study_xgb.trials_dataframe()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tr["number"] + 1, tr["value"], "o", ms=4, alpha=0.6, label="проба")
axes[0].plot(tr["number"] + 1, tr["value"].cummax(), "-", lw=1.8, label="лучшая на данный момент")
axes[0].axhline(0, color="none")
axes[0].set_xlabel("номер пробы"); axes[0].set_ylabel("AMS на val"); axes[0].legend()
axes[0].set_title("Optuna: XGBoost")
try:
    imp = optuna.importance.get_param_importances(study_xgb)
    axes[1].barh(list(imp.keys())[::-1], list(imp.values())[::-1], color="#4c72b0")
    axes[1].set_title("важность гиперпараметров (fANOVA)")
except Exception as exc:  # мало проб
    axes[1].text(0.1, 0.5, f"важность недоступна: {exc}", fontsize=8); axes[1].axis("off")
fig.tight_layout(); fig.savefig(RESULTS / "fig_optuna_xgb.png", bbox_inches="tight"); plt.show()

## 3. LightGBM: вторая реализация бустинга

Другая библиотека с тем же алгоритмическим семейством. Если результаты совпадут в пределах σ —
это аргумент, что мы измеряем свойства данных, а не особенности реализации.
Если `lightgbm` не установлен, раздел пропускается.

In [ ]:
try:
    import lightgbm as lgb
    HAVE_LGB = True
except ImportError:
    HAVE_LGB = False
    print("lightgbm не установлен — раздел пропущен (pip install lightgbm)")

best_lgb = None
if HAVE_LGB:
    def lgb_space(trial):
        return dict(
            n_estimators=3000,
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
            min_child_samples=trial.suggest_int("min_child_samples", 10, 500, log=True),
            subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        )
    def objective_lgb(trial):
        res = fit_lgb(train, val, None, lgb_space(trial), seed=42, return_model=False)
        trial.set_user_attr("best_iteration", res["best_iteration"])
        trial.set_user_attr("fit_seconds", res["fit_seconds"])
        return res["AMS_val"]
    def report_lgb(study, trial):
        print(f"проба {trial.number + 1:3d}/{N_TRIALS_LGB}: AMS_val = {trial.value:.3f} "
              f"(лучшая {study.best_value:.3f}), {trial.user_attrs['fit_seconds']:.0f} с")
        study.trials_dataframe().to_csv(RESULTS / "optuna_lgb_trials.csv", index=False)
    study_lgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study_lgb.optimize(objective_lgb, n_trials=N_TRIALS_LGB, callbacks=[report_lgb])
    best_lgb = study_lgb.best_params
    print("Лучший AMS_val (LightGBM):", round(study_lgb.best_value, 3))
    best_lgb

## 4. Лучшая модель XGBoost: оценка на test и графики для главы 3

Обучаем с лучшими параметрами, порог фиксируем по val, test открываем один раз.

In [ ]:
res_best = fit_xgb(train, val, test, dict(best_xgb, n_estimators=2000), device=DEVICE, seed=42,
                   method="XGBoost (настроенный)")
res_default = fit_xgb(train, val, test, None, device=DEVICE, seed=42, method="XGBoost (по умолчанию)")

for r in (res_default, res_best):
    print(f"{r['method']:24s} AUC_test = {r['AUC_test']:.4f}  AMS_val = {r['AMS_val']:.3f}  "
          f"AMS_test = {r['AMS_test']:.3f}  s = {r['s_test']:.0f}  b = {r['b_test']:.0f}  "
          f"деревьев = {r['best_iteration']}")

X_tr, y_tr, w_tr, _ = xy(train); X_va, y_va, w_va, _ = xy(val); X_te, y_te, w_te, _ = xy(test)
ks = ks_overtraining(res_best["scores"]["train"], y_tr, res_best["scores"]["test"], y_te)
print("KS-тест train/test (p-value):", {k: round(v["pvalue"], 3) for k, v in ks.items()})

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
plot_roc({"XGBoost по умолчанию": weighted_roc(y_te, res_default["scores"]["test"], w_te),
          "XGBoost настроенный": weighted_roc(y_te, res_best["scores"]["test"], w_te)}, ax=axes[0, 0])
axes[0, 0].set_title("ROC на test")
plot_significance_scan({"по умолчанию": significance_scan(y_va, res_default["scores"]["val"], w_va),
                        "настроенный": significance_scan(y_va, res_best["scores"]["val"], w_va)}, ax=axes[0, 1])
axes[0, 1].set_title("AMS от порога (val)")
plot_classifier_output(res_best["scores"]["train"], y_tr, w_tr, res_best["scores"]["test"], y_te, w_te, ax=axes[1, 0])
axes[1, 0].set_title("выход настроенного XGBoost: train vs test")
ev = res_best["evals"]
axes[1, 1].plot(ev["validation_0"]["logloss"], label="train")
axes[1, 1].plot(ev["validation_1"]["logloss"], label="val")
axes[1, 1].axvline(res_best["best_iteration"], color="k", ls="--", lw=0.8, label="ранняя остановка")
axes[1, 1].set_xlabel("число деревьев"); axes[1, 1].set_ylabel("logloss (взвеш.)"); axes[1, 1].legend()
axes[1, 1].set_title("кривая обучения по деревьям")
fig.tight_layout(); fig.savefig(RESULTS / "fig_bdt_best.png", bbox_inches="tight"); plt.show()

## 5. Кривая обучения: хватает ли данных

Обучаем лучшую конфигурацию на 5 %, 10 %, 25 %, 50 % и 100 % train (val и test не меняются).
Если AMS продолжает расти на 100 % — данных мало и модель выиграла бы от большего набора;
если кривая вышла на плато — ограничение уже не в статистике, а в признаках.

In [ ]:
fractions = [0.05, 0.1, 0.25, 0.5, 1.0]
curve = []
for f in fractions:
    sub = train if f == 1.0 else train.groupby("y", group_keys=False).sample(frac=f, random_state=42)
    r = fit_xgb(sub, val, test, dict(best_xgb, n_estimators=2000), device=DEVICE, seed=42, return_model=False)
    curve.append({"fraction": f, "n_train": len(sub), "AUC_test": r["AUC_test"], "AMS_test": r["AMS_test"]})
    print(f"train {100*f:5.0f} % ({len(sub):>7,}): AUC = {r['AUC_test']:.4f}, AMS_test = {r['AMS_test']:.3f}")
curve = pd.DataFrame(curve); curve.to_csv(RESULTS / "bdt_learning_curve.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(curve["n_train"], curve["AMS_test"], "o-", lw=1.8)
ax.set_xscale("log"); ax.set_xlabel("число событий в train"); ax.set_ylabel("AMS на test")
ax.set_title("кривая обучения (настроенный XGBoost)")
fig.tight_layout(); fig.savefig(RESULTS / "fig_bdt_learning_curve.png", bbox_inches="tight"); plt.show()

## 6. Повторы: разброс по разбиениям

Для каждого seed — новое разбиение 60/20/20, все методы по одному протоколу.
σ по повторам — это и есть «ошибка» в таблице результатов. Разница между методами
считается значимой, если она больше, чем ~2σ.

In [ ]:
rows = []
t0 = time.time()
for seed in SEEDS:
    tr_s, va_s, te_s = split_train_val_test(df, seed=seed)
    rows.append(dict(seed=seed, **{k: v for k, v in cut_based(va_s, te_s).items() if k != "params"}))
    rows.append(dict(seed=seed, **{k: v for k, v in fit_logreg(tr_s, va_s, te_s).items() if k != "model"}))
    r = fit_xgb(tr_s, va_s, te_s, None, device=DEVICE, seed=seed, method="XGBoost (по умолчанию)", return_model=False)
    rows.append(dict(seed=seed, **r))
    r = fit_xgb(tr_s, va_s, te_s, dict(best_xgb, n_estimators=2000), device=DEVICE, seed=seed,
                method="XGBoost (настроенный)", return_model=False)
    rows.append(dict(seed=seed, **r))
    if HAVE_LGB and best_lgb is not None:
        r = fit_lgb(tr_s, va_s, te_s, dict(best_lgb, n_estimators=3000), seed=seed,
                    method="LightGBM (настроенный)", return_model=False)
        rows.append(dict(seed=seed, **r))
    print(f"seed {seed}: готово, {(time.time() - t0) / 60:.1f} мин")

repeats = pd.DataFrame(rows)
repeats.to_csv(RESULTS / "bdt_repeats_raw.csv", index=False)
table = summarize_repeats(rows)
table.to_csv(RESULTS / "results_table.csv")
display(table)

In [ ]:
agg = repeats.groupby("method", sort=False)["AMS_test"].agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(agg.index[::-1], agg["mean"][::-1], xerr=agg["std"][::-1], color="#4c72b0", capsize=4)
ax.set_xlabel(f"AMS на test (среднее ± σ по {N_REPEATS} разбиениям)")
fig.tight_layout(); fig.savefig(RESULTS / "fig_results_table.png", bbox_inches="tight"); plt.show()

## 7. Протокол Kaggle: сравнение с лидербордом 2014 года

В конкурсе обучались на 250 тыс. событий (`KaggleSet == 't'`), публичный лидерборд считался
на 100 тыс. (`'b'`), итоговый (приватный) — на 450 тыс. (`'v'`). Столбец `KaggleWeight`
уже нормирован внутри каждой части, поэтому AMS на `'v'` с этими весами напрямую сопоставим
с приватным лидербордом. Роль val здесь играет публичная часть `'b'`.

Ориентиры (проверьте по статье Adam-Bourdarios et al., 2015 и странице конкурса):
победитель — AMS ≈ 3,81; типичный хорошо настроенный XGBoost 2014 года — ≈ 3,7.

Контроль: суммы `KaggleWeight` в каждой из частей t, b, v должны давать одинаковые
s ≈ 692 и b ≈ 411 000 — это и означает «нормировано внутри подмножества».

In [ ]:
if {"t", "b", "v"} <= set(df["KaggleSet"].unique()):
    kt, kb, kv = (df[df["KaggleSet"] == k].copy() for k in ("t", "b", "v"))
    for part in (kt, kb, kv):
        part["w"] = part["KaggleWeight"].astype(float)
    for name, part in (("t (train)", kt), ("b (public LB)", kb), ("v (private LB)", kv)):
        yl = expected_yields(part, "w")
        print(f"{name:15s}: строк {len(part):>8,}   s = {yl['s']:7.1f}   b = {yl['b']:10.1f}")

    res_k_def = fit_xgb(kt, kb, kv, None, device=DEVICE, seed=42, method="XGBoost по умолчанию", return_model=False)
    res_k_best = fit_xgb(kt, kb, kv, dict(best_xgb, n_estimators=2000), device=DEVICE, seed=42,
                         method="XGBoost настроенный", return_model=False)
    kaggle = pd.DataFrame([res_k_def, res_k_best])[["method", "AUC_test", "AMS_val", "AMS_test", "s_test", "b_test"]]
    kaggle = kaggle.rename(columns={"AMS_val": "AMS public LB (b)", "AMS_test": "AMS private LB (v)"})
    kaggle.to_csv(RESULTS / "kaggle_protocol.csv", index=False)
    display(kaggle.round(3))
else:
    print("В файле нет столбца KaggleSet с частями t/b/v — протокол Kaggle недоступен (синтетика?)")

## 8. Важность признаков (gain) — превью для `05_shap`

Gain — суммарное уменьшение функции потерь от разбиений по признаку. Грубая, но быстрая оценка.
Окончательную интерпретацию даст SHAP.

In [ ]:
imp = pd.Series(res_best["model"].get_booster().get_score(importance_type="gain"))
imp.index = [FEATURES[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k for k in imp.index]
imp = imp.sort_values(ascending=False)
imp.to_csv(RESULTS / "bdt_feature_importance_gain.csv")
fig, ax = plt.subplots(figsize=(7, 6))
top = imp.head(15)
ax.barh(top.index[::-1], top.values[::-1], color="#4c72b0")
ax.set_xlabel("gain"); ax.set_title("важность признаков, настроенный XGBoost (топ-15)")
fig.tight_layout(); fig.savefig(RESULTS / "fig_bdt_importance_gain.png", bbox_inches="tight"); plt.show()

## 9. Сохранение и вопросы для анализа

Всё, что нужно для текста, лежит в `results/`: `results_table.csv` (главная таблица, mean ± σ),
`bdt_best_params.json`, `optuna_*_trials.csv`, `bdt_learning_curve.csv`, `kaggle_protocol.csv`,
рисунки `fig_optuna_xgb.png`, `fig_bdt_best.png`, `fig_bdt_learning_curve.png`,
`fig_results_table.png`, `fig_bdt_importance_gain.png`.

**Вопросы, на которые должен ответить текст главы 3 по этим результатам:**

1. Насколько подбор гиперпараметров поднял AMS относительно параметров по умолчанию? Стоил ли
   этот прирост затраченного времени (см. `fit_seconds` в `optuna_xgb_trials.csv`)?
2. Какие гиперпараметры оказались важнее всего (график fANOVA) и почему это физически разумно
   (глубина дерева ↔ взаимодействие признаков, min_child_weight ↔ редкие области)?
3. Вышла ли кривая обучения на плато? Что это говорит о том, где предел: статистика или признаки?
4. Различимы ли методы с учётом σ? Какие пары отличаются больше чем на 2σ?
5. Совпали ли XGBoost и LightGBM в пределах σ?
6. Где настроенная модель оказалась бы в приватном лидерборде Kaggle 2014? Что отличает её
   от решения победителя (ансамбли, feature engineering, кросс-валидация)?
7. Какие 5 признаков важнее всего по gain и согласуется ли это с одиночным AUC из `01_eda`
   (напомним: масса MMC разделяет окном, а не монотонно)?

In [ ]:
summary = {
    "device": DEVICE, "n_trials_xgb": N_TRIALS, "n_repeats": N_REPEATS,
    "best_xgb_params": best_xgb, "best_xgb_ams_val": study_xgb.best_value,
    "best_lgb_params": best_lgb,
    "best_model_test": {k: res_best[k] for k in ("AUC_test", "AMS_val", "AMS_test", "Z0_test", "s_test", "b_test", "best_iteration")},
    "ks_overtraining": ks,
}
(RESULTS / "bdt_best_params.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=float))
print("Сохранено:", RESULTS / "bdt_best_params.json")